In [0]:
%run ../_shared/00_project_config

In [0]:
from pyspark.sql import functions as F

In [0]:
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

SOURCE_SYSTEM = "football-data.org"

RAW_STANDINGS_ROOT = (
    f"{FOOTBALL_DATA_ORG_RAW_ROOT}/standings"
)

BRONZE_TABLE = (
    f"{BRONZE_SCHEMA}."
    "football_data_org_standings"
)


RAW_STANDINGS_FILE_NAMES = {
    "PL": "pl_premier_league.json",
    "PD": "pd_la_liga.json",
    "BL1": "bl1_bundesliga.json",
    "SA": "sa_serie_a.json",
    "FL1": "fl1_ligue_1.json",
}

# -----------------------------------------------------------------------------
# Load football-data.org standings JSON files into Bronze
# -----------------------------------------------------------------------------
#
# Each raw JSON file contains a response envelope with:
#
#   competition
#   season
#   standings[]
#
# standings[] usually contains entries such as:
#
#   TOTAL
#   HOME
#   AWAY
#
# Each standings entry contains a table[] array.
#
# Bronze removes only these response-level arrays and produces:
#
#   one row = one team x standing type
#
# Source field names and nested team structures remain unchanged.
# -----------------------------------------------------------------------------

loaded_file_count = 0
loaded_row_count = 0


for competition_code, league in FOOTBALL_DATA_ORG_COMPETITIONS.items():

    file_name = RAW_STANDINGS_FILE_NAMES[
        competition_code
    ]

    file_path = (
        f"{RAW_STANDINGS_ROOT}/"
        f"{file_name}"
    )

    # -------------------------------------------------------------------------
    # Read raw JSON
    # -------------------------------------------------------------------------

    source_df = (
        spark.read
        .option("multiLine", "true")
        .json(file_path)
    )

    # -------------------------------------------------------------------------
    # Explode standings[]
    # -------------------------------------------------------------------------
    #
    # Keep useful response-level context:
    #
    #   competition
    #   season
    #
    # Then explode each standings table.
    # -------------------------------------------------------------------------

    standings_df = (
        source_df
        .select(
            "competition",
            "season",
            F.explode("standings").alias("standing"),
            F.col("_metadata.file_path").alias("_source_file"),
        )
    )

    # -------------------------------------------------------------------------
    # Explode standing.table[]
    # -------------------------------------------------------------------------

    bronze_df = (
        standings_df
        .select(
            "competition",
            "season",
            F.col("standing.stage").alias("stage"),
            F.col("standing.type").alias("type"),
            F.col("standing.group").alias("group"),
            F.explode("standing.table").alias("standing_row"),
            "_source_file",
        )
        .select(
            "competition",
            "season",
            "stage",
            "type",
            "group",
            "standing_row.*",
            "_source_file",
        )
        .withColumn(
            "_league",
            F.lit(league),
        )
        .withColumn(
            "_ingested_at",
            F.current_timestamp(),
        )
        .withColumn(
            "_source_system",
            F.lit(SOURCE_SYSTEM),
        )
    )

    row_count = bronze_df.count()

    # -------------------------------------------------------------------------
    # Write to Bronze Delta table
    # -------------------------------------------------------------------------

    (
        bronze_df.write
        .format("delta")
        .mode("append")
        .option(
            "mergeSchema",
            "true",
        )
        .saveAsTable(
            BRONZE_TABLE
        )
    )

    loaded_file_count += 1
    loaded_row_count += row_count

    print(
        f"Loaded football-data.org Bronze standings file "
        f"{loaded_file_count}: "
        f"{competition_code} / {league} | "
        f"rows={row_count} "
        f"-> {BRONZE_TABLE}"
    )

# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {loaded_file_count} "
    "football-data.org standings source files into Bronze. "
    f"Total standings rows loaded: {loaded_row_count}."
)